In [1]:
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward
from experiments.reward_func import verifier_reward
from experiments.common import load_aime_chat_format
from datasets import load_dataset
import os
import json

import tqdm
# dataset = load_aime_chat_format()


/data/qiuyk/exp/RL-Verifier/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# dataset = load_dataset("BytedTsinghua-SIA/DAPO-Math-17k", split="train[:2000]")
dataset = load_dataset("trl-lib/DeepMath-103K", split="train[:2000]")
print(dataset[0])
dataset = dataset.map(lambda example: {"solution": example["reward_model"]["ground_truth"]})
dataset.to_json("./dapo_math.jsonl", orient="records", lines=True)

{'data_source': 'math_dapo', 'prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.\n\nRemember to put your answer on its own line after "Answer:".', 'role': 'user'}], 'ability': 'MATH', 'reward_model': {'ground_truth': '34', 'style': 'rule-lighteval/MATH_v2'}, 'extra_info': {'index': '9a9b6eb4-a1cb-49d1-8c1e-62eaf2f74079'}}


Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 56.39ba/s]


1501411

In [ ]:
path = "./dapo_math.jsonl"

dataset = load_dataset(
    "json",
    data_files=path,
    split="train"
)

def modify_prompt(example):
    for msg in example["prompt"]:
        if msg.get("role") == "user":
            msg["content"] = msg["content"].replace(
                "The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.",
                "The last line of your response should be exactly of the form Answer: \\boxed{$Answer} where $Answer is the final answer to the problem."
            )
            msg["content"] = msg["content"].replace(
                'Remember to put your answer on its own line after "Answer:".',
                'Remember to put your final answer on its own line as Answer: \\boxed{$Answer}.'
            )
    return example

dataset = dataset.map(modify_prompt)
dataset.to_json(path, orient="records", lines=True)

Generating train split: 2000 examples [00:00, 189757.46 examples/s]
Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 77.82ba/s]


1545411

In [ ]:
print(dataset[0]["prompt"][0]["content"])

Solve the following math problem step by step. The last line of your response should be exactly of the form Answer: \boxed{$Answer} where $Answer is the final answer to the problem.

In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.

Remember to put your final answer on its own line as Answer: \boxed{$Answer}.
